In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score

def load_station(file_path, suffix):
    df = pd.read_csv(file_path, sep=';', decimal=',', encoding='utf-8-sig')
    df.columns = df.columns.str.strip()

    df['datetime'] = df['Data'].astype(str) + ' ' + df['Hora (UTC)'].astype(str)
    df = df.add_suffix(f'_{suffix}')

    return df.rename(columns={f'datetime_{suffix}': 'datetime'})

def prepare_data(df_mar, df_vil, df_jac):
    df = df_mar.merge(df_vil, on='datetime').merge(df_jac, on='datetime')

    df['delta_P'] = df['Pressao Ins. (hPa)_mar'] - df['Pressao Ins. (hPa)_vil']
    df['delta_T'] = df['Temp. Ins. (C)_mar'] - df['Temp. Ins. (C)_vil']

    features = [
        'Radiacao (KJ/m²)_jac', 'Umi. Ins. (%)_jac', 'Temp. Ins. (C)_jac',
        'delta_P', 'delta_T',
        'Vel. Vento (m/s)_mar', 'Vel. Vento (m/s)_vil'
    ]

    df = df.replace(-9999, np.nan).dropna()

    X = df[features]
    y = df['Vel. Vento (m/s)_jac']

    return X, y

def train_model(X, y):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    model = RandomForestRegressor(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)

    predictions = model.predict(X_test)
    score = r2_score(y_test, predictions)

    return model, score

if __name__ == "__main__":
    df_mar = load_station('marambaia.csv', 'mar')
    df_vil = load_station('vila_militar.csv', 'vil')
    df_jac = load_station('jacarepagua.csv', 'jac')

    X, y = prepare_data(df_mar, df_vil, df_jac)
    model, score = train_model(X, y)

    print(f"Model R²: {score:.2f}")

FileNotFoundError: [Errno 2] No such file or directory: 'marambaia.csv'